# Python Dependencies

In [1]:
!pip install -q pandas tqdm requests codecarbon neo4j pymilvus sentence-transformers tiktoken


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Env Variables

In [2]:
from pathlib import Path
from typing import Collection, Dict, List
import pandas as pd
from tqdm import tqdm
import os
import time
from codecarbon import EmissionsTracker
from pymilvus import Collection, connections
from sentence_transformers import SentenceTransformer
from typing import List
from neo4j import GraphDatabase

os.environ["ZILLIZ_URI"] = ""
os.environ["ZILLIZ_TOKEN"] = ""
os.environ["NEO4J_URI"] = ""
os.environ["NEO4J_USER"] = ""
os.environ["NEO4J_PASSWORD"] = ""
os.environ["HF_TOKEN"] = ""

print("CWD:", os.getcwd())

ModuleNotFoundError: No module named 'codecarbon'

# Utils

In [ ]:
import tiktoken
from typing import List, Any

def safe_token_count(text: str, model_name: str) -> int:
    try:
        enc = tiktoken.encoding_for_model(model_name)
        return len(enc.encode(text))
    except Exception:
        return len(text.split())

def chunk_list(data: List[Any], chunk_size: int):
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

def parse_llm_response_single(raw_text: str):
    try:
        match = re.search(r'\{.*?\}', raw_text, re.DOTALL)
        if match:
            obj = json.loads(match.group(0))
            return {
                "label": obj.get("label", "Neutral"),
                "confidence": float(obj.get("confidence", 0.0))
            }
    except Exception:
        pass

    # --- Fallback: keyword-based ---
    text = raw_text.lower()

    if "conflict" in text or "contradiction" in text:
        label = "conflict"
    elif "duplicate" in text or "entailment" in text:
        label = "duplicate"
    elif "neutral" in text:
        label = "Neutral"
    else:
        label = "Neutral"

    return {
        "label": label,
        "confidence": 0.5  # conservative default
    }

# LLM Client

In [ ]:
import time
import torch
import subprocess
from transformers import AutoTokenizer, AutoModelForCausalLM
from codecarbon import EmissionsTracker

class LocalLLMClient:
    def __init__(self, model: str, temperature: float = 0.0, device: str = None):
        self.model_name = model
        self.temperature = temperature

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(model)
        self.model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None
        )

        self.model.eval()

    def _get_gpu_power_watts(self):
        if self.device != "cuda":
            return None
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=power.draw", "--format=csv,noheader,nounits"]
            )
            return float(result.decode().strip())
        except Exception:
            return None

    def classify_batch(self, prompt: str, max_new_tokens: int = 64):
        tracker = EmissionsTracker(
            project_name=self.model_name,
            save_to_file=False,
            log_level="error"
        )

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        input_tokens = inputs["input_ids"].shape[-1]

        tracker.start()
        start_t = time.time()
        gpu_power_start = self._get_gpu_power_watts()

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=self.temperature > 0,
                temperature=self.temperature if self.temperature > 0 else None,
                pad_token_id=self.tokenizer.eos_token_id
            )

        latency = time.time() - start_t
        emissions_kg = tracker.stop()
        gpu_power_end = self._get_gpu_power_watts()

        decoded = self.tokenizer.decode(
            outputs[0][input_tokens:],
            skip_special_tokens=True
        )

        try:
            energy_kwh = tracker.final_emissions_data.energy_consumed
            avg_power_w = (energy_kwh * 3.6e6) / latency if latency > 0 else 0.0
        except Exception:
            energy_kwh = 0.0
            avg_power_w = 0.0

        return {
            "raw": decoded,
            "latency_s": latency,
            "energy_kwh": energy_kwh,
            "carbon_kg": emissions_kg,
            "avg_power_watts_estimated": avg_power_w,
            "gpu_power_watts_start": gpu_power_start,
            "gpu_power_watts_end": gpu_power_end,
            "input_tokens": input_tokens
        }


# VectorDB

In [ ]:
from pymilvus import connections, Collection
from sentence_transformers import SentenceTransformer
from typing import List

class MilvusRetriever:
    def __init__(self, collection_name: str, model_name: str):
        connections.connect(
            alias="default",
            uri=os.environ["ZILLIZ_URI"],
            token=os.environ["ZILLIZ_TOKEN"]
        )

        self.collection = Collection(collection_name)
        self.collection.load()
        self.embedder = SentenceTransformer(model_name)

    def get_candidates(self, anchor_text: str, limit: int = 5) -> List[str]:
        emb = self.embedder.encode(anchor_text, convert_to_numpy=True)

        results = self.collection.search(
            data=[emb.tolist()],
            anns_field="embedding",
            param={"metric_type": "COSINE", "params": {"nprobe": 10}},
            limit=limit,
            output_fields=["text"]
        )

        candidates = []
        for hits in results:
            for hit in hits:
                txt = hit.entity.get("text")
                if txt and txt.strip() != anchor_text.strip():
                    candidates.append(txt)

        return candidates


# Knowledge Graphs

In [ ]:
from neo4j import GraphDatabase
from typing import List

class Neo4jRetriever:
    def __init__(self, uri: str, user: str, password: str, database: str = "neo4j"):
        if GraphDatabase is None:
            raise RuntimeError("neo4j driver missing")

        self.driver = GraphDatabase.driver(
            uri,
            auth=(user, password)
        )
        self.database = database

    def get_candidates(
        self,
        anchor_text: str,
        source: str,
        limit: int = 5
    ) -> List[str]:  # type: ignore

        # Cypher query for Pure, World Vista, OpenCoss and UAV
        query = """
        MATCH (anchor:Requirement {text: $text, source: $source}) 
            -[:HAS_ACTOR|HAS_ACTION|HAS_OBJECT|HAS_CONDITION]->(ent:Entity) 
        
        MATCH (ent)<-[:HAS_ACTOR|HAS_ACTION|HAS_OBJECT|HAS_CONDITION]-(r:Requirement {source: $source}) 
        WHERE r.text <> $text WITH r, COUNT(DISTINCT ent) AS overlap_score 
        
        ORDER BY overlap_score DESC 
        RETURN r.text AS candidate_text 
        LIMIT $k

        """

        # Cypher query for IBM UAV
        query = """
        MATCH (anchor:Requirement {text: $text, source: $source})
            -[rel:HAS_ACTOR|HAS_ACTION|HAS_OBJECT|HAS_CONDITION]->(ent:Entity)
 
        MATCH (ent)<-[rel2:HAS_ACTOR|HAS_ACTION|HAS_OBJECT|HAS_CONDITION]-(r:Requirement {source: $source})
        WHERE r.text <> $text
 
        WITH r, ent, type(rel) AS rel_type
 
        MATCH (ent)<-[:HAS_ACTOR|HAS_ACTION|HAS_OBJECT|HAS_CONDITION]-(r_all:Requirement {source: $source})
        WITH r, rel_type, COUNT(DISTINCT r_all) AS entity_freq
 
        WITH r,
            SUM(
                CASE
                    WHEN rel_type = 'HAS_CONDITION'
                    THEN 2.5 / entity_freq   // BOOST CONDITIONS
                    ELSE 1.0 / entity_freq
                END
            ) AS score
 
        ORDER BY score DESC
        RETURN r.text AS candidate_text
        LIMIT $k
        """

        with self.driver.session(database=self.database) as session:
            res = session.run(
                query,
                text=anchor_text,
                source=source,
                k=limit
            )

            return [
                record["candidate_text"]
                for record in res
                if record["candidate_text"]
            ]

    def close(self):
        if self.driver:
            self.driver.close()

# Prompts

In [ ]:
ZERO_SHOT_PROMPT = """
# Task context
You are a requirements analyst.

# Goal
Given an ANCHOR requirement and ONE CANDIDATE requirement, determine whether the candidate:
- Conflicts with the Anchor
- Is Neutral to the Anchor

# Label definitions

Conflict:
The requirements cannot both be true or satisfied at the same time.
They impose incompatible or contradictory constraints.

Neutral:
They describe different, unrelated, or independent behavior.

ANCHOR:
"{anchor}"

CANDIDATE:
"{candidate}"

IMPORTANT:
You MUST return your final answer strictly in the following JSON format.
Do NOT include any text before or after the JSON.

{{
  "label": "<one of: Conflict | Neutral>",
  "confidence": <number between 0 and 1>
}}
"""

FEW_SHOT_PROMPT = """
# Task context
You are a requirements analyst.

# Goal
Given an ANCHOR requirement and ONE CANDIDATE requirement, determine whether the candidate:
- Conflicts with the Anchor
- Is Neutral to the Anchor

# Label definitions

Conflict:
The requirements cannot both be true or satisfied at the same time.
They impose incompatible or contradictory constraints.

Neutral:
They describe different, unrelated, or independent behavior.

Examples:

Example 1:
ANCHOR: The _VehicleCore_ shall assign a unique ID to each activated UAV.
CANDIDATE: The _VehicleCore_ shall assign an ID to each activated UAV.
Label: Conflict.

Example 2:
ANCHOR: When requested, the _VehicleCore_ shall transition the UAV between states according to allowed state transitions as depicted in the UAV state transition diagram.
CANDIDATE: The _MissionPlanner_ shall execute flight plans for multiple UAVs independently.
Label: Neutral.

Example 3:
ANCHOR: The _VehicleCore_ shall assign an ID to each activated UAV.
CANDIDATE: The _GCS_ shall transmit the UAV's current location to the _GCSMiddleware_.
Label: Neutral.

Now analyze the following.

ANCHOR:
"{anchor}"

CANDIDATE:
"{candidate}"

IMPORTANT:
You MUST return your final answer strictly in the following JSON format.
Do NOT include any text before or after the JSON.

{{
  "label": "<one of: Conflict | Neutral>",
  "confidence": <number between 0 and 1>
}}
"""

COT_PROMPT = """
# Task context
You are a requirements analyst.

# Goal
Given an ANCHOR requirement and ONE CANDIDATE requirement, determine whether the candidate:
- Conflicts with the Anchor
- Is Neutral to the Anchor

# Label definitions

Conflict:
The requirements cannot both be true or satisfied at the same time.
They impose incompatible or contradictory constraints.

Neutral:
They describe different, unrelated, or independent behavior.

# Thinking guidance (internal only)
Before deciding, internally:
1. Identify entities and constraints in each requirement.
2. Compare names, quantities, and conditions.
3. Decide whether the candidate conflicts with, duplicates, or is neutral to the anchor.
Do NOT reveal your reasoning.

ANCHOR:
"{anchor}"

CANDIDATE:
"{candidate}"

IMPORTANT:
You MUST return your final answer strictly in the following JSON format.
Do NOT include any text before or after the JSON.

{{
  "label": "<one of: Conflict | Neutral>",
  "confidence": <number between 0 and 1>
}}
"""


# Configurations

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR", "/content/")
FILES = {
            #  "open_coss": "open_coss_clean_pairs.csv",
            # "pure": "pure_clean_pairs.csv",
            #  "uav": "uav_clean_pairs.csv",
            #  "world_vista": "world_vista_clean_pairs.csv",
            #  "IBM_UAV": "cdn_pairs.csv"
            }

LLM_CONF = {
    "model": "mistralai/Mistral-7B-Instruct-v0.3",
    "temperature": 0.0
}

MILVUS_CONF = {
    "enabled": True,
    "collection_name": "requirements",
    "model_name": "sentence-transformers/all-mpnet-base-v2"
}

NEO4J_CONF = {
    "enabled": True,
    "uri": os.environ["NEO4J_URI"],
    "user": os.environ["NEO4J_USER"],
    "password": os.environ["NEO4J_PASSWORD"],
    "database": "neo4j"
}


# Evaluation Code

In [ ]:
from typing import Dict
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from codecarbon import EmissionsTracker

def run_dynamic_evaluation(
    data_files: Dict[str, str],
    data_dir: str,
    neo4j_conf: Dict,
    milvus_conf: Dict,
    llm_conf: Dict,
    out_dir: str = "/content/drive/MyDrive/RE_Result/",
    batch_size: int = 5,
    limit_anchors: int = 5
):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    PROMPT_STRATEGIES = {
    "zero_shot": ZERO_SHOT_PROMPT,
    "few_shot": FEW_SHOT_PROMPT,
    "cot": COT_PROMPT
    }
    # Init Managers
    kag = None
    if neo4j_conf.get("enabled"):
        kag = Neo4jRetriever(
            uri=neo4j_conf["uri"],
            user=neo4j_conf["user"],
            password=neo4j_conf["password"]
        )

    rag = None
    if milvus_conf.get("enabled"):
        rag = MilvusRetriever(
            collection_name=milvus_conf["collection_name"],
            model_name=milvus_conf["model_name"]
        )

    llm = LocalLLMClient(model = 'mistralai/Mistral-7B-Instruct-v0.3')
    tracker = EmissionsTracker(save_to_file=False, log_level="error")
    results_rows = []
    print(PROMPT_STRATEGIES)
    for ds_name, fname in data_files.items():
        path = Path(data_dir) / fname
        if not path.exists():
            print(f"Skipping {ds_name} (File not found)")
            continue

        df = pd.read_csv(path)
        unique_anchors = df['Text1'].dropna().unique()

        if limit_anchors:
            unique_anchors = unique_anchors[:limit_anchors]

        print(f"\nProcessing {ds_name}: {len(unique_anchors)} Unique Anchors...")

        for strategy_name, prompt_template in PROMPT_STRATEGIES.items():
            print(f"\nRunning strategy: {strategy_name.upper()}")

            for anchor in tqdm(unique_anchors):

                if kag:
                    tracker.start()
                    candidates = kag.get_candidates(anchor, source=ds_name, limit=2)
                    retrieval_emissions = tracker.stop()
                    retrieval_energy = tracker.final_emissions_data.energy_consumed

                    for c in candidates:
                        process_single(
                            ds_name, anchor, c,
                            f"KAG_{strategy_name}",
                            llm, results_rows,
                            retrieval_emissions, retrieval_energy,
                            prompt_template=prompt_template
                        )

                if rag:
                    tracker.start()
                    candidates = rag.get_candidates(anchor, limit=2)
                    retrieval_emissions = tracker.stop()
                    retrieval_energy = tracker.final_emissions_data.energy_consumed

                    for c in candidates:
                        process_single(
                            ds_name, anchor, c,
                            f"RAG_{strategy_name}",
                            llm, results_rows,
                            retrieval_emissions, retrieval_energy,
                            prompt_template=prompt_template
                        )

    df_out = pd.DataFrame(results_rows)
    out_csv = Path(out_dir) / "uav_dynamic_retrieval_results_rest_new_prompt.csv"

    df_out.to_csv(out_csv, index=False, float_format='%.12f')


    pd.set_option('display.float_format', lambda x: '%.12f' % x)
    pd.set_option('display.max_colwidth', None)

    print("\n" + "="*80)
    print(f"DONE. Results saved to {out_csv}")
    print("="*80)

    cols = ["pipeline", "text1", "text2", "label", "energy_kwh", "carbon_kg"]
    if not df_out.empty:
        print(df_out[cols].head(5).to_string(index=False))
    else:
        print("No results generated.")
    print("="*80)


def process_single(ds_name, anchor, candidate, pipeline_name, llm, rows_list, retreival_emissions=None, retreival_energy=None, prompt_template=None):
    prompt = prompt_template.format(
        anchor=anchor,
        candidate=candidate
    )

    metrics = llm.classify_batch(prompt) 
    prediction = parse_llm_response_single(metrics["raw"])

    rows_list.append({
        "dataset": ds_name,
        "pipeline": pipeline_name,
        "text1": anchor,
        "text2": candidate,
        "ground_truth": "N/A (Retrieved)",
        "label": prediction["label"],
        "confidence": prediction["confidence"],
        "energy_kwh": metrics["energy_kwh"],
        "carbon_kg": metrics["carbon_kg"],
        "avg_power_watts": metrics["avg_power_watts_estimated"], 
        "input_tokens": metrics["input_tokens"],
        "latency_s": metrics["latency_s"],
        "retrieval_energy_kwh": retreival_energy if retreival_energy else 0.0,
        "retrieval_carbon_kg": retreival_emissions if retreival_emissions else 0.0
    })

In [3]:
run_dynamic_evaluation(FILES, DATA_DIR, NEO4J_CONF, MILVUS_CONF, LLM_CONF, limit_anchors=None)

NameError: name 'run_dynamic_evaluation' is not defined